# What the Telescopes Cannot See
## Selection bias in the NASA exoplanet catalog

Every planet in the catalog was found by a specific method, and every method has blind spots. 
**The transit method** needs the orbit to line up with our line of sight. **The radial velocity** 
method favors heavy planets close to bright, quiet stars. In this project I measured how much 
these biases shape the catalog, correct for one of them, and look at which small planets in 
the habitable zone we have actually found so far.

**Data:** NASA Exoplanet Archive, table pscomppars (one row per confirmed planet).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

In [2]:
cols = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_orbper", "pl_orbsmax", "pl_rade", "pl_bmasse", "pl_eqt",
    "st_teff", "st_rad", "st_mass", "sy_dist", "sy_vmag",
]

query = "select " + ",".join(cols) + " from pscomppars"
url = (
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
    + query.replace(" ", "+")
    + "&format=csv"
)

try:
    raw = pd.read_csv(url)
    raw.to_csv("exoplanets_raw.csv", index=False)
except Exception as e:
    print("Download failed, trying the local copy:", e)
    raw = pd.read_csv("exoplanets_raw.csv")

print(raw.shape)
raw.head()

(6451, 14)


,pl_name,hostname,discoverymethod,disc_year,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_eqt,st_teff,st_rad,st_mass,sy_dist,sy_vmag
0,55 Cnc b,55 Cnc,Radial Velocity,1996,14.651552,0.11800,13.900,263.9785,700.00,5198.0,0.980,1.015,12.5855,5.95084
1,55 Cnc c,55 Cnc,Radial Velocity,2004,44.393600,0.24700,8.510,54.4738,499.27,5198.0,0.980,1.015,12.5855,5.95084
2,55 Cnc d,55 Cnc,Radial Velocity,2002,4799.000000,5.60000,13.000,1232.4930,104.85,5198.0,0.980,1.015,12.5855,5.95084
3,55 Cnc e,55 Cnc,Radial Velocity,2004,0.736547,0.01544,1.875,7.9900,1958.00,5172.0,0.943,0.905,12.5855,5.95084
4,55 Cnc f,55 Cnc,Radial Velocity,2007,260.580000,0.80200,7.590,44.8120,277.07,5198.0,0.980,1.015,12.5855,5.95084
